In [1]:
# [Cell 1] 설정 및 라이브러리 로드
import pdfplumber
import os
import glob
import re
from itertools import groupby
from tqdm import tqdm

# === 경로 설정 ===
BASE_DIR = "."
PDF_DIR = os.path.join(BASE_DIR, "data", "pdfs")
# 최종 결과물이 저장될 폴더
FINAL_DATA_DIR = os.path.join(BASE_DIR, "data", "processed", "05_master_data")

os.makedirs(FINAL_DATA_DIR, exist_ok=True)

print(f"✅ 설정 완료")
print(f"📂 입력: {PDF_DIR}")
print(f"📂 출력: {FINAL_DATA_DIR}")

✅ 설정 완료
📂 입력: .\data\pdfs
📂 출력: .\data\processed\05_master_data


In [2]:
# [Cell 2] 핵심 전처리 함수 정의

def clean_text_master(text: str) -> str:
    """
    [기능]
    1. 한글/영어/특수문자의 무의미한 반복 제거
    2. 숫자 고스트(222000...)를 RLE 알고리즘으로 스마트하게 복구
    3. 금액, 전화번호 등 중요 숫자는 보호
    """
    if not text: return ""

    # --- 1. 일반 텍스트 중복 제거 ---
    # 한글 반복 (벤벤벤 -> 벤)
    text = re.sub(r'([가-힣])\1+', r'\1', text)
    # 영어 단어 반복 (WordWord -> Word, 단 2글자 이상)
    text = re.sub(r'([a-zA-Z]{2,})\1+', r'\1', text)
    # 특수문자 반복 (,,, -> ,) 단, 숫자와 관련된 .,% 등은 제외
    text = re.sub(r'([^\w\s,.\-%])\1+', r'\1', text)

    # --- 2. 숫자 고스트 정밀 타격 (RLE 알고리즘) ---
    def deghost_smart(match):
        raw_str = match.group()
        
        # 안전장치: 쉼표(,)나 점(.)이 포함된 숫자는 금액/소수점이므로 건드리지 않음
        if ',' in raw_str or '.' in raw_str:
            return raw_str

        # 그룹핑: "222000" -> [('2', 3), ('0', 3)]
        groups = [(k, len(list(g))) for k, g in groupby(raw_str)]
        
        # 분석: 그룹이 2개 이상이어야 함 (단일 숫자 반복 제외)
        if len(groups) < 2: return raw_str
            
        # 균일성 체크: 모든 숫자가 동일한 횟수(N번)로 반복되는지?
        counts = [cnt for _, cnt in groups]
        first_count = counts[0]
        
        # 조건: 모든 반복 횟수가 같고, 그 횟수가 2회 이상이어야 함
        if all(c == first_count for c in counts) and first_count >= 2:
            # 압축 수행 (222000 -> 20)
            return "".join([char for char, _ in groups])
        
        return raw_str

    # 4자리 이상 연속된 숫자 덩어리만 검사 (연도, 월, 일 등)
    text = re.sub(r'\d{4,}', deghost_smart, text)

    # --- 3. 공백 및 줄바꿈 정리 ---
    text = re.sub(r'\n{3,}', '\n\n', text) # 과도한 줄바꿈 -> 2줄(문단 구분)
    text = re.sub(r'[ \t]+', ' ', text)    # 연속 공백 -> 1칸
    
    return text.strip()


def table_to_markdown(table):
    """
    pdfplumber로 추출된 표(List)를 Markdown String으로 변환
    빈 표(Ghost Table)는 자동으로 필터링하여 None 반환
    """
    if not table: return None
    
    # 전처리: None 값을 빈 문자열로, 내부 줄바꿈을 공백으로
    cleaned_table = []
    for row in table:
        cleaned_row = [str(cell).replace('\n', ' ').strip() if cell is not None else "" for cell in row]
        cleaned_table.append(cleaned_row)

    # 필터 1: 모든 셀이 비어있으면 삭제
    if not any("".join(row).strip() for row in cleaned_table):
        return None
    # 필터 2: 행이나 열이 너무 적으면 표 아님 (헤더만 있는 경우 등)
    if len(cleaned_table) < 2:
        return None

    # Markdown 변환
    md = ""
    try:
        # 헤더 생성
        headers = cleaned_table[0]
        md += "| " + " | ".join(headers) + " |\n"
        md += "| " + " | ".join(["---"] * len(headers)) + " |\n"
        # 데이터 행 생성
        for row in cleaned_table[1:]:
            md += "| " + " | ".join(row) + " |\n"
    except:
        return None 

    return md

In [3]:
# [Cell 3] 메인 실행 루프

def process_pdf_pipeline(pdf_path):
    full_text_list = []
    
    try:
        # pdfplumber로 열기
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                # 1. 텍스트 추출
                text = page.extract_text() or ""
                
                # 2. 표 추출 및 마크다운 변환
                tables = page.extract_tables()
                tables_md = []
                for table in tables:
                    md_str = table_to_markdown(table)
                    if md_str:
                        tables_md.append(md_str)
                
                # 3. 페이지 내용 결합 (텍스트 + 표)
                # 표는 문맥 흐름을 끊지 않게 페이지 하단에 배치 (RAG 검색 유리)
                page_content = text
                if tables_md:
                    page_content += "\n\n[참고: 표 데이터]\n" + "\n".join(tables_md)
                
                full_text_list.append(page_content)
                
    except Exception as e:
        print(f"⚠️ 읽기 오류 ({os.path.basename(pdf_path)}): {e}")
        return ""

    # 전체 페이지 결합
    combined_text = "\n\n".join(full_text_list)
    
    # 4. 최종 스마트 정제 (구조, 중복, 고스트 제거)
    final_cleaned = clean_text_master(combined_text)
    
    return final_cleaned

# === 실행 ===
pdf_files = sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
print(f"🚀 전처리 파이프라인 시작! (대상: {len(pdf_files)}개 파일)")

success_cnt = 0
skip_cnt = 0

pbar = tqdm(pdf_files)
for pdf_path in pbar:
    filename = os.path.basename(pdf_path).replace(".pdf", ".txt")
    save_path = os.path.join(FINAL_DATA_DIR, filename)
    
    # 이어하기 기능: 이미 처리된 파일은 건너뜀
    if os.path.exists(save_path):
        skip_cnt += 1
        continue
        
    # 처리 수행
    result_text = process_pdf_pipeline(pdf_path)
    
    # 내용이 유의미하게 있을 때만 저장 (빈 스캔본 제외)
    if len(result_text) > 50:
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(result_text)
        success_cnt += 1
        pbar.set_description(f"✅ 저장: {filename}")
    else:
        pbar.set_description(f"⚠️ 내용없음: {filename}")

print("\n" + "="*50)
print(f"🎉 모든 작업 완료!")
print(f"   - 처리 성공: {success_cnt}개")
print(f"   - 건너뜀 (이미 완료): {skip_cnt}개")
print(f"📁 결과물 폴더: {FINAL_DATA_DIR}")

🚀 전처리 파이프라인 시작! (대상: 100개 파일)


✅ 저장: BioIN_의료기기산업 종합정보시스템(정보관리기관) 기능개선 사업(2차).txt:   6%|▌         | 6/100 [02:10<31:56, 20.39s/it]   /it]  Cannot set gray non-stroke color because /'Pa1' is an invalid float value
Cannot set gray non-stroke color because /'Pa2' is an invalid float value
Cannot set gray non-stroke color because /'Pa3' is an invalid float value
Cannot set gray non-stroke color because /'Pa4' is an invalid float value
✅ 저장: 세종테크노파크_세종테크노파크 인사정보 전산시스템 구축 용역 입찰공.txt:  44%|████▍     | 44/100 [19:28<38:50, 41.62s/it]  t]          Cannot set gray non-stroke color because /'Pa1' is an invalid float value
Cannot set gray non-stroke color because /'Pa2' is an invalid float value
Cannot set gray non-stroke color because /'Pa3' is an invalid float value
Cannot set gray non-stroke color because /'Pa4' is an invalid float value
Cannot set gray non-stroke color because /'Pa5' is an invalid float value
Cannot set gray non-stroke color because /'Pa6' is an invalid float value
Cannot set gray non-stroke color because /


🎉 모든 작업 완료!
   - 처리 성공: 100개
   - 건너뜀 (이미 완료): 0개
📁 결과물 폴더: .\data\processed\05_master_data


In [4]:
# 셀 15: [Version Control] 05번 데이터를 보정하여 06번 폴더에 저장
import os
import glob
import re
from tqdm import tqdm

# === 경로 설정 ===
BASE_DIR = "."
SOURCE_DIR = os.path.join(BASE_DIR, "data", "processed", "05_master_data")
NEW_DIR = os.path.join(BASE_DIR, "data", "processed", "06_text_merged")

# 폴더 생성
os.makedirs(NEW_DIR, exist_ok=True)

# === 표 보호 줄바꿈 보정 함수 (Logic by Noah) ===
def fix_broken_lines_safe(text: str) -> str:
    if not text: return ""

    lines = text.split('\n')
    new_lines = []
    text_buffer = []

    def flush_buffer():
        """모아둔 텍스트 라인을 공백으로 이어 붙여서 추가"""
        if text_buffer:
            # 버퍼에 있는 줄들을 공백으로 연결
            joined = " ".join(text_buffer)
            # 다중 공백 정리 ("  " -> " ")
            joined = re.sub(r'[ ]+', ' ', joined)
            new_lines.append(joined)
            text_buffer.clear()

    for line in lines:
        stripped = line.strip()

        # 1. 빈 줄 (문단 구분) -> 버퍼 비우고 빈 줄 유지
        if not stripped:
            flush_buffer()
            new_lines.append("") 
            continue

        # 2. 표(Markdown) 및 메타 태그 보호
        # '|' 로 시작하거나 '---', '[' 로 시작하는 줄은 합치지 않음
        if stripped.startswith('|') or stripped.startswith('[') or stripped.startswith('---'):
            flush_buffer() # 표 나오기 전 텍스트 마무리
            new_lines.append(line) # 표 라인은 원본 그대로 (줄바꿈 유지)
        
        # 3. 일반 텍스트 -> 버퍼에 담기 (나중에 합쳐짐)
        else:
            text_buffer.append(stripped)

    # 마지막 버퍼 비우기
    flush_buffer()

    return "\n".join(new_lines)

# === 실행 로직 ===
files = sorted(glob.glob(os.path.join(SOURCE_DIR, "*.txt")))
print(f"🚀 데이터 후처리 및 이동 시작...")
print(f"   📂 원본: {SOURCE_DIR}")
print(f"   📂 타겟: {NEW_DIR}")

processed_count = 0
for file_path in tqdm(files):
    filename = os.path.basename(file_path)
    save_path = os.path.join(NEW_DIR, filename)
    
    # 1. 원본 읽기
    with open(file_path, "r", encoding="utf-8") as f:
        original_text = f.read()
    
    # 2. 변환 수행 (Table-Safe Line Merge)
    fixed_text = fix_broken_lines_safe(original_text)
    
    # 3. 새로운 폴더에 저장
    with open(save_path, "w", encoding="utf-8") as f:
        f.write(fixed_text)
    
    processed_count += 1

print(f"\n✅ 작업 완료! 총 {processed_count}개 파일이 '06_text_merged' 폴더에 생성되었습니다.")

# === 결과 비교 (검증) ===
# 첫 번째 파일로 전/후 차이 확인
if files:
    filename = os.path.basename(files[0])
    src = os.path.join(SOURCE_DIR, filename)
    dst = os.path.join(NEW_DIR, filename)
    
    print("\n🔎 [비교 검증: 05 vs 06]")
    with open(src, "r", encoding="utf-8") as f: 
        print(f"🔻 [05 원본] (상위 150자)\n{f.read()[:150]}")
    print("-" * 30)
    with open(dst, "r", encoding="utf-8") as f: 
        print(f"🔻 [06 보정] (상위 150자)\n{f.read()[:150]}")

🚀 데이터 후처리 및 이동 시작...
   📂 원본: .\data\processed\05_master_data
   📂 타겟: .\data\processed\06_text_merged


100%|██████████| 100/100 [00:01<00:00, 89.19it/s]


✅ 작업 완료! 총 100개 파일이 '06_text_merged' 폴더에 생성되었습니다.

🔎 [비교 검증: 05 vs 06]
🔻 [05 원본] (상위 150자)
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업
- (복수의결권주식, 스톡옵션, 성과조건부주식) -
제안요청서
2024. 03.

목 차
1. 추진개요 · 3
2. 추진방안 · 5
3. 추진내용 · 9
4. 제안요청내용 · 24
5. 입찰관련사항 · 78
6.
------------------------------
🔻 [06 보정] (상위 150자)
2024년 ｢벤처확인종합관리시스템 기능 고도화｣ 용역사업 - (복수의결권주식, 스톡옵션, 성과조건부주식) - 제안요청서 2024. 03.

목 차 1. 추진개요 · 3 2. 추진방안 · 5 3. 추진내용 · 9 4. 제안요청내용 · 24 5. 입찰관련사항 · 78 6.
